In [ ]:
#test d'extraction rameau a l'aide d'une base de vecteurs
import sys
!{sys.executable} -m pip install pinecone-client
!{sys.executable} -m  install sentence-transformers
!{sys.executable} -m  install datasets


In [7]:
import pandas as pd
import numpy as np

In [36]:
import pinecone

# connect to pinecone environment
pinecone.init(
    api_key="VOTRE_CLE_PINECONE",
    environment="us-west4-gcp"  # find next to API key in console
)

In [37]:
# pour re-indexer il faut au prealable supprimer l'index sur l'interface d'admin
index_name = 'extreme-ml'

# check if the extreme-ml index exists
if index_name not in pinecone.list_indexes():
    # create the index if it does not exist
    pinecone.create_index(
        index_name,
        dimension=384,
        metric="cosine"
    )

# connect to extreme-ml index we created
index = pinecone.Index(index_name)

In [80]:
#chargemnt fichier d'entrainement
df1 = pd.read_csv('export_rameau.csv') 
df=df1.drop(columns=['PPN'])
df['RAMEAU']=df['RAMEAU'].str.replace(' ','_')


In [81]:
df.head(20)

,RAMEAU,TITRE
0,Psychologie,Le corporel nouvelles approches en psychos...
1,Médecine,Le corporel nouvelles approches en psychos...
2,Psychologie,Le corporel nouvelles approches en psychos...
3,Psychologie,Le corporel nouvelles approches en psychos...
4,Médecine,Le corporel nouvelles approches en psychos...
5,Psychologie,Le corporel nouvelles approches en psychos...
6,Psychologie,"Psychologie pour l'enseignant [L3. Master"" e..."
7,Éducation,"Psychologie pour l'enseignant [L3. Master"" e..."
8,Psychologie,"Psychologie pour l'enseignant [L3. Master"" e..."
9,Éducation,"Psychologie pour l'enseignant [L3. Master"" e..."


In [82]:
df = (df.groupby(['TITRE'])
      .agg({'RAMEAU': lambda x: x.tolist()})
      .reset_index())
df.head()

,TITRE,RAMEAU
0,'Iore iti ē = Petite souris = Little mouse,"[Sciences_de_l'information, Sciences_sociales,..."
1,'Silence' in painting [exposition] London ...,[Peinture]
2,(Pas) tout sur la mère actes du colloque,"[Sciences_sociales, Médecine, Psychologie, Sci..."
3,+ de 160 nouvelles phrases pour s'amuser à bie...,"[Jeux, Poésie]"
4,... Et personne ne voulut le croire feux de ...,[Religion]


In [83]:
#chargement model
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the model from huggingface
model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2',
    device=device
)
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
  (2): Normalize()
)

In [84]:
import pandas as pd

# Create embeddings
encoded_articles = model.encode(df['TITRE'].tolist(), show_progress_bar=True)
# add the embeddings to our dataframe
df['content_vector'] = pd.Series(encoded_articles.tolist())

Batches:   0%|          | 0/2355 [00:00<?, ?it/s]

In [85]:
import numpy as np

# Explode the target indicator column
df_explode = df.explode('RAMEAU')
#df_explode.head(5)
# Group by label and define a unique vector for each label
label_vectors = df_explode.groupby('RAMEAU').agg(mean=('content_vector', lambda x: np.vstack(x).mean(axis=0).tolist()))
label_vectors['target'] = label_vectors.index
label_vectors.columns = ['content_vector', 'label']

label_vectors.sample(10)



,content_vector,label
RAMEAU,,
Sciences_de_la_Terre,"[-0.011459733294792027, 0.04343000251089038, 0...",Sciences_de_la_Terre
Sciences,"[-0.023033748476785677, 0.029011640602409298, ...",Sciences
Agriculture,"[-0.014863918746520658, 0.02562995392782238, -...",Agriculture
Dessin__arts_décoratifs__artisanat_d'art,"[-0.02454482269132317, 0.04893136464424714, 0....",Dessin__arts_décoratifs__artisanat_d'art
Chimie__minéralogie__cristallographie,"[-0.030562817275175642, -0.012824867700062826,...",Chimie__minéralogie__cristallographie
Biologie_des_procaryotes,"[-0.017064584125069567, -0.019476761587118974,...",Biologie_des_procaryotes
Sculpture,"[-0.01840709024738582, 0.06862409910136132, -0...",Sculpture
Sciences_médicales_et_paramédicales,"[0.00294956392908104, 0.025131750309325032, -0...",Sciences_médicales_et_paramédicales
Sciences_sociales,"[-0.010488521537724345, 0.04264556642181804, -...",Sciences_sociales


In [86]:
print(label_vectors.size)



198


In [87]:
from tqdm.auto import tqdm
#alimentation index 
# we will use batches of 256
batch_size = 256

for i in tqdm(range(0, len(label_vectors), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(label_vectors))
    # extract batch
    batch = label_vectors.iloc[i:i_end]
    # select embeddings for batch
    emb = batch["content_vector"].tolist()
    # get metadata
    meta = [{"label": l} for l in batch["label"]]
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

  0%|          | 0/1 [00:00<?, ?it/s]

In [88]:
# check that we have all vectors in index
index.describe_index_stats()

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 99}},
 'total_vector_count': 99}

In [89]:
#chargement fichier de test
df_test = pd.read_csv('data/test_rameau_export.csv') 
df_test=df_test.drop(columns=['PPN'])
df_test['RAMEAU']=df_test['RAMEAU'].str.replace(' ','_')
df_test = (df_test.groupby(['TITRE'])
      .agg({'RAMEAU': lambda x: x.tolist()})
      .reset_index())
df_test.head()

,TITRE,RAMEAU
0,Analyse du comportement mécanique d'une prothè...,"[Médecine, Biologie]"
1,Analyse par la méthode des éléments finis du c...,"[Mathématiques, Médecine, Médecine]"
2,Analyse structurale de la biogenèse de la peti...,"[Biologie, Physique, Biologie]"
3,Annales corrigées de l'internat en odontologie...,"[Médecine, Éducation, Pharmacie]"
4,Approche expérimentale et théorique de la ciné...,"[Chimie, Chimie]"


In [92]:
from pprint import pprint
def select_test_article(index):
    print("choisi un element du fichier de  test:")
    # select the article associated with index from test split
    article = df_test.iloc[index]
    # print test article data
    data = {"Titre": article.TITRE[:1000],  "Original Labels": list(article.RAMEAU)}
    pprint(data)
    return article
def query_pinecone(article, top_k=3):
    print("predict rameau:")
    # Create embeddings for test articles
    xq = model.encode(article.TITRE).tolist()
    # query pinecone for labels
    results = index.query(xq, top_k=top_k, include_metadata=True)
    print(results)
    # select only the labels from result and print
    labels = [res["metadata"]["label"] for res in results.matches]
    pprint({"Predicted Labels": labels})

In [90]:
article = select_test_article(2)

Query Article:
{'Original Labels': ['Biologie', 'Physique', 'Biologie'],
 'Titre': 'Analyse structurale de la biogenèse de la petite sous-unité '
          'ribosomique eucaryote par cryo-microscopie électronique et analyse '
          "d'images"}


In [93]:
query_pinecone(article, top_k=10)

predict rameau:
{'matches': [{'id': '17',
              'metadata': {'label': 'Biologie_des_procaryotes'},
              'score': 0.364192367,
              'values': []},
             {'id': '20',
              'metadata': {'label': 'Chimie'},
              'score': 0.311991513,
              'values': []},
             {'id': '16',
              'metadata': {'label': 'Biologie'},
              'score': 0.299540401,
              'values': []},
             {'id': '84',
              'metadata': {'label': 'Sciences_de_la_vie'},
              'score': 0.271862715,
              'values': []},
             {'id': '21',
              'metadata': {'label': 'Chimie__minéralogie__cristallographie'},
              'score': 0.271373481,
              'values': []},
             {'id': '62',
              'metadata': {'label': 'Paléontologie'},
              'score': 0.260460645,
              'values': []},
             {'id': '18',
              'metadata': {'label': 'Botanique'},
          